# World Cup Prediction — Poisson + LightGBM Optuna

Objectif : modèle focalisé et performant pour prédire `p_home_win`, `p_draw`, `p_away_win`.

Ce notebook fait :

1. chargement et harmonisation des données ;
2. intégration des odds historiques + odds WC2026 fournies ;
3. feature engineering anti-leakage : Elo, forme récente, H2H, performance vs confédération adverse, repos, calendrier, attaque/défense ;
4. backtest complet sur WC2022 ;
5. entraînement final ;
6. inférence group stage WC2026 ;
7. simulation optionnelle du bracket à partir de `knockout_slots.csv`.

Modèles utilisés uniquement :

- **Poisson** pour les buts attendus et probabilités de score ;
- **LightGBM tuned avec Optuna** pour la classification multiclass.

> Important : les colonnes post-match comme tirs, xG, cartons, corners ne sont jamais utilisées comme features prédictives.


In [1]:
# Optional installs. Uncomment if needed.
!pip install -q lightgbm optuna scikit-learn pandas numpy scipy openpyxl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 6.0 MB/s eta 0:00:00


In [10]:
from pathlib import Path
from collections import defaultdict, deque
import math
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.metrics import log_loss, accuracy_score, brier_score_loss
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import PoissonRegressor

try:
    import lightgbm as lgb
except Exception as e:
    raise ImportError('Install LightGBM first: pip install lightgbm') from e

try:
    import optuna
except Exception as e:
    raise ImportError('Install Optuna first: pip install optuna') from e

DATA_DIR = Path('./')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_2022_PATH = DATA_DIR / 'test_wc2022.csv'
RESULTS_PATH = DATA_DIR / 'results.csv'
ELO_PATH = DATA_DIR / 'eloratings.csv'
WC_XLSX_PATH = DATA_DIR / 'WorldCup2026.xlsx'
WC2026_FIXTURES_WITH_ODDS_PATH = DATA_DIR / 'wc2026_group_fixtures_with_odds_fixed.csv'
KNOCKOUT_SLOTS_PATH = DATA_DIR / 'knockout_slots.csv'

RANDOM_STATE = 42
N_OPTUNA_TRIALS = 80  # increase to 200+ if you have time
LGBM_NUM_BOOST_ROUND = 5000
EARLY_STOPPING_ROUNDS = 150
POISSON_MAX_GOALS = 10

# Backtest style: predict all WC2022 matches as if we are before the tournament.
WC2022_START = pd.Timestamp('2022-11-20')
WC2026_START = pd.Timestamp('2026-06-11')


In [11]:
# Check the cleaned odds file created from your pasted odds.
wc2026_odds = pd.read_csv(WC2026_FIXTURES_WITH_ODDS_PATH)
print(wc2026_odds.shape)
display(wc2026_odds.head())
print('Missing odds:', wc2026_odds['H_odds'].isna().sum())
assert wc2026_odds['H_odds'].notna().all(), 'Some WC2026 group fixtures did not receive odds.'


(72, 27)


,match_id,group,date_utc,date,phase,neutral,venue,tournament,home_team_original,away_team_original,...,market_p_away_mean,market_favorite_prob,market_underdog_prob,market_home_minus_away,market_draw_gap,market_entropy,has_market_odds,datetime_local,home_team_raw,away_team_raw
0,1,A,2026-06-11 19:00:00+00:00,2026-06-11,Group,1,"Estadio Azteca, Mexico City",FIFA World Cup,Mexico,South Africa,...,0.121106,0.663594,0.121106,0.542488,-0.448295,0.858436,1,2026-06-11 21:00:00,Mexico,South Africa
1,2,A,2026-06-12 02:00:00+00:00,2026-06-12,Group,1,"Estadio Akron, Guadalajara",FIFA World Cup,South Korea,UEFA Playoff D,...,0.342214,0.362955,0.294831,0.020740,-0.068124,1.094902,1,2026-06-12 04:00:00,South Korea,Czech Republic
2,3,B,2026-06-12 19:00:00+00:00,2026-06-12,Group,1,"BMO Field, Toronto",FIFA World Cup,Canada,UEFA Playoff A,...,0.210826,0.527066,0.210826,0.316239,-0.264957,1.016709,1,2026-06-12 21:00:00,Canada,Bosnia & Herzegovina
3,4,D,2026-06-13 01:00:00+00:00,2026-06-13,Group,1,"SoFi Stadium, Los Angeles",FIFA World Cup,USA,Paraguay,...,0.244019,0.472488,0.244019,0.228469,-0.188995,1.055798,1,2026-06-13 03:00:00,USA,Paraguay
4,5,D,2026-06-13 04:00:00+00:00,2026-06-13,Group,1,"BC Place, Vancouver",FIFA World Cup,Australia,UEFA Playoff C,...,0.521387,0.521387,0.214348,-0.307039,-0.257122,1.021373,1,2026-06-14 06:00:00,Australia,Turkey


Missing odds: 0


In [12]:
TEAM_NAME_MAP = {
    'USA': 'United States',
    'U.S.A.': 'United States',
    'United States of America': 'United States',
    'United States': 'United States',
    'South Korea': 'Korea Republic',
    'Korea Republic': 'Korea Republic',
    'Korea DPR': 'North Korea',
    'Iran': 'Iran',
    'IR Iran': 'Iran',
    "Côte d\'Ivoire": 'Ivory Coast',
    "Cote d\'Ivoire": 'Ivory Coast',
    'Ivory Coast': 'Ivory Coast',
    'Curaçao': 'Curacao',
    'Curacao': 'Curacao',
    'Cabo Verde': 'Cape Verde',
    'Cape Verde': 'Cape Verde',
    'Bosnia & Herzegovina': 'Bosnia and Herzegovina',
    'Bosnia-Herzegovina': 'Bosnia and Herzegovina',
    'Bosnia and Herzegovina': 'Bosnia and Herzegovina',
    'D.R. Congo': 'DR Congo',
    'Congo DR': 'DR Congo',
    'Democratic Republic of Congo': 'DR Congo',
    'DR Congo': 'DR Congo',
    'Czechia': 'Czech Republic',
    'Czech Republic': 'Czech Republic',
    'Türkiye': 'Turkey',
}

PLACEHOLDER_MAP_2026 = {
    'UEFA Playoff D': 'Czech Republic',
    'UEFA Playoff A': 'Bosnia and Herzegovina',
    'UEFA Playoff C': 'Turkey',
    'UEFA Playoff B': 'Sweden',
    'FIFA Playoff 2': 'Iraq',
    'FIFA Playoff 1': 'DR Congo',
}

MANUAL_CONF = {
    'Mexico': 'CONCACAF', 'South Africa': 'CAF', 'Korea Republic': 'AFC', 'Czech Republic': 'UEFA',
    'Canada': 'CONCACAF', 'Bosnia and Herzegovina': 'UEFA', 'United States': 'CONCACAF', 'Paraguay': 'CONMEBOL',
    'Australia': 'AFC', 'Turkey': 'UEFA', 'Qatar': 'AFC', 'Switzerland': 'UEFA', 'Brazil': 'CONMEBOL',
    'Morocco': 'CAF', 'Haiti': 'CONCACAF', 'Scotland': 'UEFA', 'Germany': 'UEFA', 'Curacao': 'CONCACAF',
    'Netherlands': 'UEFA', 'Japan': 'AFC', 'Ivory Coast': 'CAF', 'Ecuador': 'CONMEBOL', 'Sweden': 'UEFA',
    'Tunisia': 'CAF', 'Spain': 'UEFA', 'Cape Verde': 'CAF', 'Belgium': 'UEFA', 'Egypt': 'CAF',
    'Saudi Arabia': 'AFC', 'Uruguay': 'CONMEBOL', 'Iran': 'AFC', 'New Zealand': 'OFC', 'France': 'UEFA',
    'Senegal': 'CAF', 'Iraq': 'AFC', 'Norway': 'UEFA', 'Argentina': 'CONMEBOL', 'Algeria': 'CAF',
    'Austria': 'UEFA', 'Jordan': 'AFC', 'Portugal': 'UEFA', 'DR Congo': 'CAF', 'England': 'UEFA',
    'Croatia': 'UEFA', 'Ghana': 'CAF', 'Panama': 'CONCACAF', 'Uzbekistan': 'AFC', 'Colombia': 'CONMEBOL',
}

HOSTS_2026 = {'United States', 'Mexico', 'Canada'}


def canonical_team(x):
    if pd.isna(x):
        return x
    x = str(x).strip()
    x = PLACEHOLDER_MAP_2026.get(x, x)
    return TEAM_NAME_MAP.get(x, x)


def canonicalize_team_cols(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = df[c].map(canonical_team)
    return df


def add_result(df):
    df = df.copy()
    if 'result' not in df.columns:
        df['result'] = np.where(df['home_score'] > df['away_score'], 'H', np.where(df['home_score'] < df['away_score'], 'A', 'D'))
    return df


def tournament_importance(t):
    t = '' if pd.isna(t) else str(t).lower()
    if 'world cup' in t and 'qualification' not in t:
        return 5
    if 'continental' in t or 'euro' in t or 'african cup' in t or 'asian cup' in t or 'copa america' in t:
        return 4
    if 'qualification' in t or 'qualifier' in t:
        return 3
    if 'friendly' in t:
        return 1
    return 2


In [13]:
def parse_float_safe(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        x = x.replace(',', '.').replace('−','-').strip()
        if x in ['', '-', '—']:
            return np.nan
    try:
        return float(x)
    except Exception:
        return np.nan


def coalesce_columns(df, target_col, candidates):
    existing = [c for c in candidates if c in df.columns]
    vals = pd.Series(np.nan, index=df.index)
    for c in existing:
        vals = vals.combine_first(df[c].map(parse_float_safe))
    df[target_col] = vals
    return df


def implied_probs_from_odds(h, d, a):
    vals = np.array([h, d, a], dtype=float)
    if np.any(pd.isna(vals)) or np.any(vals <= 1.0):
        return np.array([np.nan, np.nan, np.nan])
    inv = 1.0 / vals
    return inv / inv.sum()


def add_implied_prob_triplet(df, prefix, h, d, a):
    df = df.copy()
    if not all(c in df.columns for c in [h, d, a]):
        for side in ['home','draw','away']:
            df[f'odds_p_{side}_{prefix}'] = np.nan
        return df
    p = df.apply(lambda r: implied_probs_from_odds(r[h], r[d], r[a]), axis=1)
    p = np.vstack(p.values) if len(p) else np.empty((0, 3))
    df[f'odds_p_home_{prefix}'] = p[:, 0]
    df[f'odds_p_draw_{prefix}'] = p[:, 1]
    df[f'odds_p_away_{prefix}'] = p[:, 2]
    return df


def add_market_consensus_features(df):
    df = df.copy()
    home_cols = [c for c in df.columns if c.startswith('odds_p_home_')]
    draw_cols = [c for c in df.columns if c.startswith('odds_p_draw_')]
    away_cols = [c for c in df.columns if c.startswith('odds_p_away_')]
    if not home_cols:
        for c in ['market_p_home_mean','market_p_draw_mean','market_p_away_mean','market_favorite_prob','market_underdog_prob','market_home_minus_away','market_draw_gap','market_entropy']:
            df[c] = np.nan
        df['has_market_odds'] = 0
        return df
    df['market_p_home_mean'] = df[home_cols].mean(axis=1, skipna=True)
    df['market_p_draw_mean'] = df[draw_cols].mean(axis=1, skipna=True)
    df['market_p_away_mean'] = df[away_cols].mean(axis=1, skipna=True)
    s = df[['market_p_home_mean','market_p_draw_mean','market_p_away_mean']].sum(axis=1)
    valid = s.notna() & (s > 0)
    for c in ['market_p_home_mean','market_p_draw_mean','market_p_away_mean']:
        df.loc[valid, c] = df.loc[valid, c] / s[valid]
    probs = df[['market_p_home_mean','market_p_draw_mean','market_p_away_mean']]
    df['has_market_odds'] = valid.astype(int)
    df['market_favorite_prob'] = probs.max(axis=1)
    df['market_underdog_prob'] = probs.min(axis=1)
    df['market_home_minus_away'] = df['market_p_home_mean'] - df['market_p_away_mean']
    df['market_draw_gap'] = df['market_p_draw_mean'] - probs[['market_p_home_mean','market_p_away_mean']].max(axis=1)
    eps = 1e-12
    df['market_entropy'] = -(probs.fillna(0).clip(eps, 1) * np.log(probs.fillna(0).clip(eps, 1))).sum(axis=1)
    return df


def load_worldcup_excel(path):
    if not Path(path).exists():
        return pd.DataFrame()
    xls = pd.ExcelFile(path)
    frames = []
    for sheet in xls.sheet_names:
        df = pd.read_excel(path, sheet_name=sheet)
        df = df.copy()
        df['sheet'] = sheet
        rename = {'Date': 'date', 'Home': 'home_team', 'Away': 'away_team', 'HG': 'home_score', 'AG': 'away_score', 'HGFT': 'home_score', 'AGFT': 'away_score'}
        df = df.rename(columns={k:v for k,v in rename.items() if k in df.columns})
        if not {'date','home_team','away_team'}.issubset(df.columns):
            continue
        df = canonicalize_team_cols(df, ['home_team','away_team'])
        df['date'] = pd.to_datetime(df['date'], format='mixed', errors='coerce')
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    wc = pd.concat(frames, ignore_index=True, sort=False)
    wc = coalesce_columns(wc, 'H_avg_unified', ['H-Avg','H_Avg','H Avg','AvgH'])
    wc = coalesce_columns(wc, 'D_avg_unified', ['D-Avg','D_Avg','D Avg','AvgD'])
    wc = coalesce_columns(wc, 'A_avg_unified', ['A-Avg','A_Avg','A Avg','AvgA'])
    wc = coalesce_columns(wc, 'H_max_unified', ['H-Max','H_Max','H Max','MaxH'])
    wc = coalesce_columns(wc, 'D_max_unified', ['D-Max','D_Max','D Max','MaxD'])
    wc = coalesce_columns(wc, 'A_max_unified', ['A-Max','A_Max','A Max','MaxA'])
    wc = coalesce_columns(wc, 'bet365_H_unified', ['bet365-H','B365H','Bet365H'])
    wc = coalesce_columns(wc, 'bet365_D_unified', ['bet365-D','B365D','Bet365D'])
    wc = coalesce_columns(wc, 'bet365_A_unified', ['bet365-A','B365A','Bet365A'])
    wc = coalesce_columns(wc, 'pinny_H_unified', ['Pinny-H','Pinnacle-H','PSH'])
    wc = coalesce_columns(wc, 'pinny_D_unified', ['Pinny-D','Pinnacle-D','PSD'])
    wc = coalesce_columns(wc, 'pinny_A_unified', ['Pinny-A','Pinnacle-A','PSA'])
    wc = coalesce_columns(wc, 'betfair_H_unified', ['Betfair_Exch-H','Betfair-H','BFH'])
    wc = coalesce_columns(wc, 'betfair_D_unified', ['Betfair_Exch-D','Betfair-D','BFD'])
    wc = coalesce_columns(wc, 'betfair_A_unified', ['Betfair_Exch-A','Betfair-A','BFA'])
    for prefix, cols in {
        'avg': ('H_avg_unified','D_avg_unified','A_avg_unified'),
        'max': ('H_max_unified','D_max_unified','A_max_unified'),
        'bet365': ('bet365_H_unified','bet365_D_unified','bet365_A_unified'),
        'pinny': ('pinny_H_unified','pinny_D_unified','pinny_A_unified'),
        'betfair': ('betfair_H_unified','betfair_D_unified','betfair_A_unified'),
    }.items():
        wc = add_implied_prob_triplet(wc, prefix, *cols)
    wc = add_market_consensus_features(wc)
    keep = ['date','home_team','away_team','home_score','away_score','sheet','market_p_home_mean','market_p_draw_mean','market_p_away_mean','market_favorite_prob','market_underdog_prob','market_home_minus_away','market_draw_gap','market_entropy','has_market_odds']
    keep = [c for c in keep if c in wc.columns]
    return wc[keep].dropna(subset=['date','home_team','away_team']).copy()


In [14]:
def load_base_history():
    train = pd.read_csv(TRAIN_PATH)
    train['source'] = 'train'
    results = pd.read_csv(RESULTS_PATH)
    results['source'] = 'results'
    wc_excel = load_worldcup_excel(WC_XLSX_PATH)
    wc_excel['source'] = 'wc_excel'

    # Align schemas.
    for df in [train, results, wc_excel]:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df['home_team'] = df['home_team'].map(canonical_team)
        df['away_team'] = df['away_team'].map(canonical_team)
        if 'neutral' not in df.columns:
            df['neutral'] = True
        if 'tournament' not in df.columns:
            df['tournament'] = df.get('sheet', 'Unknown')

    cols = ['date','home_team','away_team','home_score','away_score','tournament','neutral','source']
    history = pd.concat([train[cols], results[cols], wc_excel[cols]], ignore_index=True, sort=False)
    history = history.dropna(subset=['date','home_team','away_team','home_score','away_score'])
    history['home_score'] = history['home_score'].astype(float)
    history['away_score'] = history['away_score'].astype(float)
    history = add_result(history)
    history['neutral'] = history['neutral'].astype(int)
    history['tournament_importance'] = history['tournament'].map(tournament_importance)

    # Drop duplicates, preferring rows with richer / competition data. Same date + teams + scores is enough.
    history = history.sort_values(['date','source']).drop_duplicates(['date','home_team','away_team','home_score','away_score'], keep='last')
    history = history.sort_values('date').reset_index(drop=True)
    return history, wc_excel

history_all, wc_excel = load_base_history()
print(history_all.shape)
display(history_all.tail())
print(wc_excel.groupby('sheet')['has_market_odds'].mean())


(49496, 10)


,date,home_team,away_team,home_score,away_score,tournament,neutral,source,result,tournament_importance
49491,2026-05-31,Czech Republic,Kosovo,2.0,1.0,Friendly,0,results,H,1
49492,2026-05-31,Cape Verde,Serbia,3.0,0.0,Friendly,1,results,H,1
49493,2026-05-31,Brazil,Panama,6.0,2.0,Friendly,0,results,H,1
49494,2026-05-31,Germany,Finland,4.0,0.0,Friendly,0,results,H,1
49495,2026-05-31,Japan,Iceland,1.0,0.0,Friendly,0,results,H,1


sheet
WorldCup2014              1.000000
WorldCup2018              1.000000
WorldCup2022              1.000000
WorldCup2026Qualifiers    0.998875
Name: has_market_odds, dtype: float64


In [15]:
# Confederation mapping learned from train + manual mapping.
train_raw = pd.read_csv(TRAIN_PATH)
train_raw = canonicalize_team_cols(train_raw, ['home_team','away_team'])
TEAM_TO_CONF = {}
for side in ['home','away']:
    tmp = train_raw[[f'{side}_team', f'{side}_conf']].dropna().drop_duplicates()
    TEAM_TO_CONF.update(dict(zip(tmp[f'{side}_team'], tmp[f'{side}_conf'])))
TEAM_TO_CONF.update(MANUAL_CONF)

def get_conf(team):
    return TEAM_TO_CONF.get(team, 'UNK')

missing_conf = sorted(set(history_all['home_team']).union(history_all['away_team']) - set(TEAM_TO_CONF))
print('Missing conf count:', len(missing_conf))
print(missing_conf[:50])


Missing conf count: 258
['Abkhazia', 'Afghanistan', 'Alderney', 'Ambazonia', 'American Samoa', 'Andalusia', 'Andorra', 'Angola', 'Anguilla', 'Antigua and Barbuda', 'Arameans Suryoye', 'Armenia', 'Artsakh', 'Aruba', 'Asturias', 'Aymara', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barawa', 'Barbados', 'Basque Country', 'Belarus', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Biafra', 'Bonaire', 'Botswana', 'British Virgin Islands', 'Brittany', 'Brunei', 'Burundi', 'Cambodia', 'Canary Islands', 'Cascadia', 'Catalonia', 'Cayman Islands', 'Central Africa', 'Central African Republic', 'Central Spain', 'Chad', 'Chagos Islands', 'Chameria', 'Chechnya', 'China', 'China PR', 'Chinese Taipei']


In [16]:
# Merge market odds into any feature frame by date + team pair.
MARKET_FEATURES = ['market_p_home_mean','market_p_draw_mean','market_p_away_mean','market_favorite_prob','market_underdog_prob','market_home_minus_away','market_draw_gap','market_entropy','has_market_odds']


def build_market_table():
    wc_mkt = wc_excel.copy()
    wc_mkt = canonicalize_team_cols(wc_mkt, ['home_team','away_team'])
    wc_mkt['date'] = pd.to_datetime(wc_mkt['date']).dt.normalize()
    wc2026 = pd.read_csv(WC2026_FIXTURES_WITH_ODDS_PATH)
    wc2026['date'] = pd.to_datetime(wc2026['date']).dt.normalize()
    wc2026 = canonicalize_team_cols(wc2026, ['home_team','away_team'])
    keep = ['date','home_team','away_team'] + MARKET_FEATURES
    market = pd.concat([wc_mkt[keep], wc2026[keep]], ignore_index=True, sort=False)
    market = market.drop_duplicates(['date','home_team','away_team'], keep='last')
    return market

MARKET_TABLE = build_market_table()
print(MARKET_TABLE.shape, MARKET_TABLE['has_market_odds'].mean())
display(MARKET_TABLE.tail())


def merge_market_features(df):
    df = df.copy()
    df['date_norm'] = pd.to_datetime(df['date']).dt.normalize()
    out = df.merge(
        MARKET_TABLE.rename(columns={'date':'date_norm'}),
        on=['date_norm','home_team','away_team'],
        how='left'
    ).drop(columns=['date_norm'])
    for c in MARKET_FEATURES:
        if c not in out.columns:
            out[c] = np.nan
    out['has_market_odds'] = out['has_market_odds'].fillna(0).astype(int)
    return out


(1153, 12) 0.9991326973113617


,date,home_team,away_team,market_p_home_mean,market_p_draw_mean,market_p_away_mean,market_favorite_prob,market_underdog_prob,market_home_minus_away,market_draw_gap,market_entropy,has_market_odds
1148,2026-06-27,Croatia,Ghana,0.550459,0.256881,0.192661,0.550459,0.192661,0.357798,-0.293578,0.995042,1
1149,2026-06-27,Colombia,Portugal,0.270035,0.286157,0.443808,0.443808,0.270035,-0.173773,-0.157651,1.072108,1
1150,2026-06-27,DR Congo,Uzbekistan,0.377069,0.291371,0.331560,0.377069,0.291371,0.045508,-0.085697,1.093097,1
1151,2026-06-28,Algeria,Austria,0.285538,0.294190,0.420272,0.420272,0.285538,-0.134734,-0.126082,1.082152,1
1152,2026-06-28,Jordan,Argentina,0.056873,0.130654,0.812473,0.812473,0.056873,-0.755599,-0.681818,0.597688,1


In [22]:
def add_external_elo_features(df):
    df = df.copy()

    if not ELO_PATH.exists():
        df["ext_elo_home"] = np.nan
        df["ext_elo_away"] = np.nan
        df["ext_elo_diff"] = np.nan
        return df

    elo = pd.read_csv(ELO_PATH)

    elo["date"] = pd.to_datetime(elo["date"], errors="coerce")
    elo["team"] = elo["team"].map(canonical_team)

    elo = (
        elo
        .dropna(subset=["date", "team", "rating"])
        .copy()
    )

    # Critical for merge_asof: sort by date first, then by team.
    elo = elo.sort_values(["date", "team"]).reset_index(drop=True)

    def merge_side(base, side):
        left = base[["date", f"{side}_team"]].copy()
        left["row_id"] = np.arange(len(left))

        left = left.rename(columns={f"{side}_team": "team"})
        left["date"] = pd.to_datetime(left["date"], errors="coerce")
        left["team"] = left["team"].map(canonical_team)

        left = (
            left
            .dropna(subset=["date", "team"])
            .sort_values(["date", "team"])
            .reset_index(drop=True)
        )

        m = safe_merge_asof(
            left=left,
            right=elo[["date", "team", "rating"]],
            left_on="date",
            right_on="date",
            left_by="team",
            right_by="team",
            direction="backward",
            suffixes=("", "_elo"),
        )

        # Restore original row order.
        out = (
            m[["row_id", "rating"]]
            .sort_values("row_id")
            .set_index("row_id")["rating"]
        )

        # Reindex to full df length to handle rows dropped because of NaN keys.
        return out.reindex(np.arange(len(base))).values

    df["ext_elo_home"] = merge_side(df, "home")
    df["ext_elo_away"] = merge_side(df, "away")
    df["ext_elo_diff"] = df["ext_elo_home"] - df["ext_elo_away"]

    return df

In [25]:
def safe_merge_asof(
    left,
    right,
    left_on,
    right_on,
    left_by=None,
    right_by=None,
    direction="backward",
    suffixes=("", "_right"),
):
    """
    Robust wrapper around pandas.merge_asof.

    Important:
    pandas merge_asof requires both frames to be sorted by the 'on' key.
    When using by/group keys, sorting by date first is usually safer than sorting by group first.
    """

    left = left.copy()
    right = right.copy()

    # Drop rows with null merge keys
    left_required = [left_on]
    right_required = [right_on]

    if left_by is not None:
        left_required += [left_by] if isinstance(left_by, str) else list(left_by)

    if right_by is not None:
        right_required += [right_by] if isinstance(right_by, str) else list(right_by)

    left = left.dropna(subset=left_required)
    right = right.dropna(subset=right_required)

    # Force datetime
    left[left_on] = pd.to_datetime(left[left_on], errors="coerce")
    right[right_on] = pd.to_datetime(right[right_on], errors="coerce")

    left = left.dropna(subset=[left_on])
    right = right.dropna(subset=[right_on])

    # Critical: sort by time first
    left_sort_cols = [left_on]
    right_sort_cols = [right_on]

    if left_by is not None:
        left_sort_cols += [left_by] if isinstance(left_by, str) else list(left_by)

    if right_by is not None:
        right_sort_cols += [right_by] if isinstance(right_by, str) else list(right_by)

    left = left.sort_values(left_sort_cols).reset_index(drop=True)
    right = right.sort_values(right_sort_cols).reset_index(drop=True)

    return pd.merge_asof(
        left,
        right,
        left_on=left_on,
        right_on=right_on,
        left_by=left_by,
        right_by=right_by,
        direction=direction,
        suffixes=suffixes,
    )

In [26]:
class FeatureBuilder:
    def __init__(self, k_base=30.0, elo_home_adv=45.0):
        self.k_base = k_base
        self.elo_home_adv = elo_home_adv
        self.elo = defaultdict(lambda: 1500.0)
        self.team_matches = defaultdict(lambda: deque(maxlen=80))
        self.h2h = defaultdict(lambda: deque(maxlen=10))
        self.last_date = {}

    def _team_window(self, team, n, opp_conf=None):
        games = list(self.team_matches[team])
        if opp_conf is not None:
            games = [g for g in games if g['opp_conf'] == opp_conf]
        games = games[-n:]
        if not games:
            return {f'last{n}_points_avg': 1.0, f'last{n}_win_rate': 0.33, f'last{n}_draw_rate': 0.28,
                    f'last{n}_gd_avg': 0.0, f'last{n}_gf_avg': 1.2, f'last{n}_ga_avg': 1.2,
                    f'last{n}_clean_sheet_rate': 0.25, f'last{n}_failed_score_rate': 0.25, f'last{n}_n': 0}
        pts = np.array([g['points'] for g in games], dtype=float)
        gd = np.array([g['gf'] - g['ga'] for g in games], dtype=float)
        gf = np.array([g['gf'] for g in games], dtype=float)
        ga = np.array([g['ga'] for g in games], dtype=float)
        return {f'last{n}_points_avg': pts.mean(), f'last{n}_win_rate': np.mean(pts == 3), f'last{n}_draw_rate': np.mean(pts == 1),
                f'last{n}_gd_avg': gd.mean(), f'last{n}_gf_avg': gf.mean(), f'last{n}_ga_avg': ga.mean(),
                f'last{n}_clean_sheet_rate': np.mean(ga == 0), f'last{n}_failed_score_rate': np.mean(gf == 0), f'last{n}_n': len(games)}

    def _days_since(self, team, date):
        if team not in self.last_date:
            return 999.0
        return float((date - self.last_date[team]).days)

    def _matches_recent_days(self, team, date, days):
        cutoff = date - pd.Timedelta(days=days)
        return sum(1 for g in self.team_matches[team] if g['date'] >= cutoff)

    def _h2h_features(self, home, away):
        key = tuple(sorted([home, away]))
        games = list(self.h2h[key])[-3:]
        if not games:
            return {'h2h_last3_home_points_avg': 1.0, 'h2h_last3_home_win_rate': 0.33, 'h2h_last3_draw_rate': 0.28, 'h2h_last3_home_gd_avg': 0.0, 'h2h_last3_n': 0}
        vals = []
        draws = []
        gds = []
        for g in games:
            if g['home_ref'] == home:
                pts = g['home_points']
                gd = g['home_gd']
            else:
                pts = 3 - g['home_points'] if g['home_points'] in [0,3] else 1
                gd = -g['home_gd']
            vals.append(pts)
            draws.append(pts == 1)
            gds.append(gd)
        vals = np.array(vals)
        return {'h2h_last3_home_points_avg': vals.mean(), 'h2h_last3_home_win_rate': np.mean(vals == 3), 'h2h_last3_draw_rate': np.mean(draws), 'h2h_last3_home_gd_avg': np.mean(gds), 'h2h_last3_n': len(games)}

    def make_features_for_match(self, row):
        date = pd.Timestamp(row['date'])
        h, a = row['home_team'], row['away_team']
        hc, ac = get_conf(h), get_conf(a)
        neutral = int(row.get('neutral', 1))
        home_adv = 0.0 if neutral else self.elo_home_adv
        feats = {
            'date': date, 'home_team': h, 'away_team': a,
            'home_conf': hc, 'away_conf': ac,
            'neutral': neutral,
            'same_conf': int(hc == ac),
            'tournament_importance': tournament_importance(row.get('tournament', '')),
            'is_world_cup': int('world cup' in str(row.get('tournament', '')).lower() and 'qualification' not in str(row.get('tournament', '')).lower()),
            'is_qualifier': int('qualification' in str(row.get('tournament', '')).lower() or 'qualifier' in str(row.get('tournament', '')).lower()),
            'is_friendly': int('friendly' in str(row.get('tournament', '')).lower()),
            'home_is_host_2026': int(h in HOSTS_2026),
            'away_is_host_2026': int(a in HOSTS_2026),
            'elo_home': self.elo[h],
            'elo_away': self.elo[a],
            'elo_diff': self.elo[h] + home_adv - self.elo[a],
            'home_days_since_last': self._days_since(h, date),
            'away_days_since_last': self._days_since(a, date),
            'rest_days_diff': self._days_since(h, date) - self._days_since(a, date),
            'home_matches_last30': self._matches_recent_days(h, date, 30),
            'away_matches_last30': self._matches_recent_days(a, date, 30),
            'home_matches_last90': self._matches_recent_days(h, date, 90),
            'away_matches_last90': self._matches_recent_days(a, date, 90),
        }
        for n in [3,5,10,20]:
            hw = self._team_window(h, n)
            aw = self._team_window(a, n)
            for k, v in hw.items(): feats[f'home_{k}'] = v
            for k, v in aw.items(): feats[f'away_{k}'] = v
            # differential features
            for metric in ['points_avg','win_rate','draw_rate','gd_avg','gf_avg','ga_avg','clean_sheet_rate','failed_score_rate','n']:
                feats[f'diff_last{n}_{metric}'] = feats[f'home_last{n}_{metric}'] - feats[f'away_last{n}_{metric}']
        # Recent performance against opponent's confederation.
        for n in [5,10]:
            hwc = self._team_window(h, n, opp_conf=ac)
            awc = self._team_window(a, n, opp_conf=hc)
            for metric in ['points_avg','win_rate','gd_avg','gf_avg','ga_avg','n']:
                feats[f'home_vs_awayconf_last{n}_{metric}'] = hwc[f'last{n}_{metric}']
                feats[f'away_vs_homeconf_last{n}_{metric}'] = awc[f'last{n}_{metric}']
                feats[f'diff_vs_conf_last{n}_{metric}'] = hwc[f'last{n}_{metric}'] - awc[f'last{n}_{metric}']
        feats.update(self._h2h_features(h, a))
        return feats

    def update(self, row):
        if pd.isna(row.get('home_score')) or pd.isna(row.get('away_score')):
            return
        date = pd.Timestamp(row['date'])
        h, a = row['home_team'], row['away_team']
        hg, ag = float(row['home_score']), float(row['away_score'])
        hc, ac = get_conf(h), get_conf(a)
        hp = 3 if hg > ag else 1 if hg == ag else 0
        ap = 3 if ag > hg else 1 if hg == ag else 0
        self.team_matches[h].append({'date': date, 'opponent': a, 'opp_conf': ac, 'gf': hg, 'ga': ag, 'points': hp})
        self.team_matches[a].append({'date': date, 'opponent': h, 'opp_conf': hc, 'gf': ag, 'ga': hg, 'points': ap})
        self.last_date[h] = date
        self.last_date[a] = date
        key = tuple(sorted([h, a]))
        self.h2h[key].append({'date': date, 'home_ref': h, 'home_points': hp, 'home_gd': hg - ag})
        # Elo update.
        neutral = int(row.get('neutral', 1))
        home_adv = 0.0 if neutral else self.elo_home_adv
        rh, ra = self.elo[h], self.elo[a]
        eh = 1.0 / (1.0 + 10 ** ((ra - (rh + home_adv)) / 400.0))
        sh = 1.0 if hg > ag else 0.5 if hg == ag else 0.0
        gd = abs(hg - ag)
        gd_mult = 1.0 if gd <= 1 else 1.0 + math.log(gd)
        k = self.k_base * gd_mult * (0.75 + 0.1 * tournament_importance(row.get('tournament', '')))
        delta = k * (sh - eh)
        self.elo[h] += delta
        self.elo[a] -= delta


def build_training_features(history):
    history = history.copy().sort_values('date').reset_index(drop=True)
    fb = FeatureBuilder()
    feats = []
    for _, row in history.iterrows():
        f = fb.make_features_for_match(row)
        f['home_score'] = row['home_score']
        f['away_score'] = row['away_score']
        f['result'] = row['result']
        f['match_id'] = row.get('match_id', np.nan)
        feats.append(f)
        fb.update(row)
    out = pd.DataFrame(feats)
    out = merge_market_features(out)
    out = add_external_elo_features(out)
    out['ext_elo_diff'] = out['ext_elo_diff'].fillna(out['elo_diff'])
    out['ext_elo_home'] = out['ext_elo_home'].fillna(out['elo_home'])
    out['ext_elo_away'] = out['ext_elo_away'].fillna(out['elo_away'])
    return out


def build_scoring_features(history, matches_to_score, freeze_at_min_score_date=True):
    history = history.copy()
    matches = matches_to_score.copy()
    matches['date'] = pd.to_datetime(matches['date'], errors='coerce')
    matches = canonicalize_team_cols(matches, ['home_team','away_team'])
    if 'tournament' not in matches.columns:
        matches['tournament'] = 'FIFA World Cup'
    if 'neutral' not in matches.columns:
        matches['neutral'] = 1
    cutoff = matches['date'].min() if freeze_at_min_score_date else None
    if cutoff is not None:
        hist_used = history[pd.to_datetime(history['date']) < cutoff].copy()
    else:
        hist_used = history.copy()
    hist_used = hist_used.sort_values('date')
    fb = FeatureBuilder()
    for _, row in hist_used.iterrows():
        fb.update(row)
    feats = []
    for _, row in matches.iterrows():
        f = fb.make_features_for_match(row)
        f['match_id'] = row.get('match_id', np.nan)
        f['group'] = row.get('group', np.nan)
        f['phase'] = row.get('phase', np.nan)
        f['venue'] = row.get('venue', np.nan)
        feats.append(f)
    out = pd.DataFrame(feats)
    out = merge_market_features(out)
    out = add_external_elo_features(out)
    out['ext_elo_diff'] = out['ext_elo_diff'].fillna(out['elo_diff'])
    out['ext_elo_home'] = out['ext_elo_home'].fillna(out['elo_home'])
    out['ext_elo_away'] = out['ext_elo_away'].fillna(out['elo_away'])
    return out


In [27]:
# Build features. This can take a few minutes because it iterates chronologically.
features_all = build_training_features(history_all)
print(features_all.shape)
display(features_all.tail())


(49496, 188)


,date,home_team,away_team,home_conf,away_conf,neutral,same_conf,tournament_importance,is_world_cup,is_qualifier,...,market_p_away_mean,market_favorite_prob,market_underdog_prob,market_home_minus_away,market_draw_gap,market_entropy,has_market_odds,ext_elo_home,ext_elo_away,ext_elo_diff
49491,2026-05-31,Poland,Ukraine,UEFA,UEFA,0,1,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,1829.816284,1826.416072,48.400212
49492,2026-05-31,Switzerland,Jordan,UEFA,AFC,0,0,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,1981.013438,1784.278741,241.734697
49493,2026-05-31,Singapore,Mongolia,UNK,UNK,0,1,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,1296.031469,977.347295,363.684174
49494,2026-05-31,United States,Senegal,CONCACAF,CAF,0,0,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,1877.662862,1928.648284,-5.985422
49495,2026-05-31,Japan,Iceland,AFC,UEFA,0,0,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,2029.489802,1656.689084,417.800718


In [28]:
TARGET_MAP = {'H': 0, 'D': 1, 'A': 2}
INV_TARGET_MAP = {0: 'H', 1: 'D', 2: 'A'}

NON_FEATURE_COLS = {'date','home_team','away_team','home_conf','away_conf','result','home_score','away_score','match_id','group','phase','venue'}
FEATURE_COLS = [c for c in features_all.columns if c not in NON_FEATURE_COLS and pd.api.types.is_numeric_dtype(features_all[c])]
print('n_features:', len(FEATURE_COLS))
print(FEATURE_COLS[:50])

# Optional: categorical encodings for teams/confs can help LGBM.
# To keep inference stable, this notebook focuses on numeric engineered features.


n_features: 179
['neutral', 'same_conf', 'tournament_importance', 'is_world_cup', 'is_qualifier', 'is_friendly', 'home_is_host_2026', 'away_is_host_2026', 'elo_home', 'elo_away', 'elo_diff', 'home_days_since_last', 'away_days_since_last', 'rest_days_diff', 'home_matches_last30', 'away_matches_last30', 'home_matches_last90', 'away_matches_last90', 'home_last3_points_avg', 'home_last3_win_rate', 'home_last3_draw_rate', 'home_last3_gd_avg', 'home_last3_gf_avg', 'home_last3_ga_avg', 'home_last3_clean_sheet_rate', 'home_last3_failed_score_rate', 'home_last3_n', 'away_last3_points_avg', 'away_last3_win_rate', 'away_last3_draw_rate', 'away_last3_gd_avg', 'away_last3_gf_avg', 'away_last3_ga_avg', 'away_last3_clean_sheet_rate', 'away_last3_failed_score_rate', 'away_last3_n', 'diff_last3_points_avg', 'diff_last3_win_rate', 'diff_last3_draw_rate', 'diff_last3_gd_avg', 'diff_last3_gf_avg', 'diff_last3_ga_avg', 'diff_last3_clean_sheet_rate', 'diff_last3_failed_score_rate', 'diff_last3_n', 'home_las

In [29]:
def poisson_score_matrix(lambda_home, lambda_away, max_goals=10):
    # Independent Poisson score probabilities.
    goals = np.arange(max_goals + 1)
    ph = np.exp(-lambda_home) * np.power(lambda_home, goals) / np.array([math.factorial(g) for g in goals])
    pa = np.exp(-lambda_away) * np.power(lambda_away, goals) / np.array([math.factorial(g) for g in goals])
    mat = np.outer(ph, pa)
    mat = mat / mat.sum()
    return mat


def score_matrix_to_outcome_probs(mat):
    p_home = np.tril(mat, -1).sum()  # rows home goals > cols away goals
    p_draw = np.trace(mat)
    p_away = np.triu(mat, 1).sum()
    p = np.array([p_home, p_draw, p_away])
    return p / p.sum()


def fit_poisson_models(train_df, feature_cols):
    X = train_df[feature_cols]
    y_home = train_df['home_score'].clip(lower=0)
    y_away = train_df['away_score'].clip(lower=0)
    home_model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), PoissonRegressor(alpha=1e-4, max_iter=1000))
    away_model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), PoissonRegressor(alpha=1e-4, max_iter=1000))
    home_model.fit(X, y_home)
    away_model.fit(X, y_away)
    return home_model, away_model


def predict_poisson(models, df, feature_cols):
    home_model, away_model = models
    X = df[feature_cols]
    lh = np.clip(home_model.predict(X), 0.05, 5.5)
    la = np.clip(away_model.predict(X), 0.05, 5.5)
    probs = np.vstack([score_matrix_to_outcome_probs(poisson_score_matrix(h, a, POISSON_MAX_GOALS)) for h, a in zip(lh, la)])
    return probs, lh, la


In [30]:
def temporal_train_valid_split(df, valid_start=None):
    df = df.sort_values('date').copy()
    if valid_start is None:
        # Last 20% by date order.
        split_idx = int(len(df) * 0.80)
        cutoff = df.iloc[split_idx]['date']
    else:
        cutoff = pd.Timestamp(valid_start)
    train_idx = df['date'] < cutoff
    valid_idx = df['date'] >= cutoff
    return df[train_idx].copy(), df[valid_idx].copy(), cutoff


def tune_lgbm_optuna(train_df, feature_cols, n_trials=N_OPTUNA_TRIALS, valid_start='2019-01-01'):
    tr, va, cutoff = temporal_train_valid_split(train_df, valid_start=valid_start)
    X_tr, y_tr = tr[feature_cols], tr['result'].map(TARGET_MAP)
    X_va, y_va = va[feature_cols], va['result'].map(TARGET_MAP)

    def objective(trial):
        params = {
            'objective': 'multiclass',
            'num_class': 3,
            'metric': 'multi_logloss',
            'verbosity': -1,
            'boosting_type': 'gbdt',
            'seed': RANDOM_STATE,
            'feature_pre_filter': False,
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.08, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 8, 96),
            'max_depth': trial.suggest_int('max_depth', 2, 9),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 15, 180),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.55, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.55, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 8),
            'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
            'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 30.0, log=True),
            'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 0.5),
        }
        dtr = lgb.Dataset(X_tr, label=y_tr)
        dva = lgb.Dataset(X_va, label=y_va, reference=dtr)
        model = lgb.train(
            params,
            dtr,
            valid_sets=[dva],
            num_boost_round=LGBM_NUM_BOOST_ROUND,
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)]
        )
        pred = model.predict(X_va, num_iteration=model.best_iteration)
        return log_loss(y_va, pred, labels=[0,1,2])

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print('valid cutoff:', cutoff)
    print('best value:', study.best_value)
    print('best params:', study.best_params)
    return study


def fit_lgbm_final(train_df, feature_cols, params, valid_start=None):
    y = train_df['result'].map(TARGET_MAP)
    base_params = {
        'objective': 'multiclass', 'num_class': 3, 'metric': 'multi_logloss', 'verbosity': -1,
        'seed': RANDOM_STATE, 'boosting_type': 'gbdt'
    }
    base_params.update(params)
    if valid_start is not None:
        tr, va, _ = temporal_train_valid_split(train_df, valid_start=valid_start)
        dtr = lgb.Dataset(tr[feature_cols], label=tr['result'].map(TARGET_MAP))
        dva = lgb.Dataset(va[feature_cols], label=va['result'].map(TARGET_MAP), reference=dtr)
        model = lgb.train(base_params, dtr, valid_sets=[dva], num_boost_round=LGBM_NUM_BOOST_ROUND,
                          callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)])
        best_iter = model.best_iteration
    else:
        dtr = lgb.Dataset(train_df[feature_cols], label=y)
        # Conservative number if we train on all data. You can increase after backtest.
        best_iter = 1000
        model = lgb.train(base_params, dtr, num_boost_round=best_iter)
    return model


def blend_probs(p_lgbm, p_poisson, weight_lgbm=0.72):
    p = weight_lgbm * p_lgbm + (1 - weight_lgbm) * p_poisson
    p = np.clip(p, 0.015, 0.97)
    return p / p.sum(axis=1, keepdims=True)


def optimize_blend_weight(y_true, p_lgbm, p_poisson):
    best = (999, None)
    for w in np.linspace(0, 1, 101):
        p = blend_probs(p_lgbm, p_poisson, w)
        ll = log_loss(y_true, p, labels=[0,1,2])
        if ll < best[0]:
            best = (ll, w)
    return best[1], best[0]


In [40]:
# Backtest on WC2022.
# Train only on matches before the tournament and predict all 64 matches as pre-tournament fixtures.

hist_pre_2022 = history_all[history_all["date"] < WC2022_START].copy()
features_pre_2022 = features_all[features_all["date"] < WC2022_START].copy()

# Load test WC2022 fixture file.
test_2022 = pd.read_csv(TEST_2022_PATH)
test_2022["date"] = pd.to_datetime(test_2022["date"], errors="coerce").dt.normalize()
test_2022 = canonicalize_team_cols(test_2022, ["home_team", "away_team"])
test_2022["tournament"] = "FIFA World Cup"
test_2022["neutral"] = 1

# Build features exactly as pre-tournament inference.
test_2022_feat = build_scoring_features(
    hist_pre_2022,
    test_2022,
    freeze_at_min_score_date=True
)


def attach_actual_results_to_features_allow_reverse(features_df, results_df):
    features_df = features_df.copy()
    results_df = results_df.copy()

    features_df["date"] = pd.to_datetime(features_df["date"], errors="coerce").dt.normalize()
    results_df["date"] = pd.to_datetime(results_df["date"], errors="coerce").dt.normalize()

    features_df = canonicalize_team_cols(features_df, ["home_team", "away_team"])
    results_df = canonicalize_team_cols(results_df, ["home_team", "away_team"])

    def result_label(h, a):
        if pd.isna(h) or pd.isna(a):
            return np.nan
        if h > a:
            return "H"
        if h < a:
            return "A"
        return "D"

    def invert_result(x):
        if x == "H":
            return "A"
        if x == "A":
            return "H"
        if x == "D":
            return "D"
        return np.nan

    actual = results_df[
        ["date", "home_team", "away_team", "home_score", "away_score"]
    ].dropna(subset=["date", "home_team", "away_team"]).copy()

    actual["source_result"] = actual.apply(
        lambda r: result_label(r["home_score"], r["away_score"]),
        axis=1
    )

    # Direct merge: same home/away order.
    direct_actual = actual.rename(columns={
        "home_score": "actual_home_score_direct",
        "away_score": "actual_away_score_direct",
        "source_result": "result_direct",
    })

    out = features_df.drop(
        columns=["result", "home_score", "away_score"],
        errors="ignore"
    ).merge(
        direct_actual,
        on=["date", "home_team", "away_team"],
        how="left",
    )

    # Reverse merge: source has the teams in the opposite order.
    reverse_actual = actual.rename(columns={
        "home_team": "away_team",
        "away_team": "home_team",
        "home_score": "source_home_score_reverse",
        "away_score": "source_away_score_reverse",
        "source_result": "source_result_reverse",
    })

    out = out.merge(
        reverse_actual,
        on=["date", "home_team", "away_team"],
        how="left",
    )

    out["result"] = out["result_direct"]
    out["home_score"] = out["actual_home_score_direct"]
    out["away_score"] = out["actual_away_score_direct"]

    reverse_mask = out["result"].isna() & out["source_result_reverse"].notna()

    out.loc[reverse_mask, "result"] = (
        out.loc[reverse_mask, "source_result_reverse"].map(invert_result)
    )
    out.loc[reverse_mask, "home_score"] = out.loc[reverse_mask, "source_away_score_reverse"]
    out.loc[reverse_mask, "away_score"] = out.loc[reverse_mask, "source_home_score_reverse"]

    debug_cols = [
        "result_direct",
        "actual_home_score_direct",
        "actual_away_score_direct",
        "source_home_score_reverse",
        "source_away_score_reverse",
        "source_result_reverse",
    ]

    out = out.drop(columns=[c for c in debug_cols if c in out.columns], errors="ignore")

    return out


# Attach true WC2022 results, allowing reversed home/away.
results_raw = pd.read_csv(RESULTS_PATH)

test_2022_eval = attach_actual_results_to_features_allow_reverse(
    features_df=test_2022_feat,
    results_df=results_raw,
)

missing = test_2022_eval[test_2022_eval["result"].isna()]

print("Missing actual results:", len(missing))
display(missing[["match_id", "date", "home_team", "away_team"]])

assert test_2022_eval["result"].notna().all(), "Some WC2022 actual results were not found."

y_2022 = test_2022_eval["result"].map(TARGET_MAP).values

print(
    test_2022_eval[
        [
            "match_id",
            "date",
            "home_team",
            "away_team",
            "result",
            "home_score",
            "away_score",
            "has_market_odds",
        ]
    ].head()
)

print("WC2022 market odds coverage:", test_2022_eval["has_market_odds"].mean())

Missing actual results: 0


,match_id,date,home_team,away_team


   match_id       date    home_team away_team result  home_score  away_score  \
0     10001 2022-11-20        Qatar   Ecuador      A         0.0         2.0   
1     10002 2022-11-25        Qatar   Senegal      A         1.0         3.0   
2     10003 2022-11-25  Netherlands   Ecuador      D         1.0         1.0   
3     10004 2022-11-29      Ecuador   Senegal      A         1.0         2.0   
4     10005 2022-11-29  Netherlands     Qatar      H         2.0         0.0   

   has_market_odds  
0                1  
1                1  
2                1  
3                1  
4                1  
WC2022 market odds coverage: 0.984375


,date,home_team,away_team,home_conf,away_conf,neutral,same_conf,tournament_importance,is_world_cup,is_qualifier,...,market_draw_gap,market_entropy,has_market_odds,ext_elo_home,ext_elo_away,ext_elo_diff,date_norm,result,home_score,away_score
5,2022-11-29,Senegal,Ecuador,CAF,CONMEBOL,1,0,5,1,0,...,NaN,NaN,0,1820.26223,1908.356413,-88.094183,2022-11-29,NaN,NaN,NaN


In [ ]:
# Tune LightGBM on pre-WC2022 history, then fit and evaluate.
study_2022 = tune_lgbm_optuna(features_pre_2022, FEATURE_COLS, n_trials=N_OPTUNA_TRIALS, valid_start='2019-01-01')
lgbm_2022 = fit_lgbm_final(features_pre_2022, FEATURE_COLS, study_2022.best_params, valid_start='2019-01-01')
poisson_2022 = fit_poisson_models(features_pre_2022, FEATURE_COLS)

p_lgbm_2022 = lgbm_2022.predict(test_2022_eval[FEATURE_COLS], num_iteration=lgbm_2022.best_iteration)
p_poisson_2022, lambda_h_2022, lambda_a_2022 = predict_poisson(poisson_2022, test_2022_eval, FEATURE_COLS)

# Optimize blend on the WC2022 backtest only for diagnostics.
# For a clean final model, you can use this weight or optimize on the internal validation.
best_w_2022, best_ll_2022 = optimize_blend_weight(y_2022, p_lgbm_2022, p_poisson_2022)
p_blend_2022 = blend_probs(p_lgbm_2022, p_poisson_2022, best_w_2022)

print('WC2022 LightGBM logloss:', log_loss(y_2022, p_lgbm_2022, labels=[0,1,2]), 'acc:', accuracy_score(y_2022, p_lgbm_2022.argmax(axis=1)))
print('WC2022 Poisson  logloss:', log_loss(y_2022, p_poisson_2022, labels=[0,1,2]), 'acc:', accuracy_score(y_2022, p_poisson_2022.argmax(axis=1)))
print('WC2022 Blend    logloss:', log_loss(y_2022, p_blend_2022, labels=[0,1,2]), 'acc:', accuracy_score(y_2022, p_blend_2022.argmax(axis=1)), 'w_lgbm:', best_w_2022)

backtest_2022 = test_2022_eval[['match_id','date','home_team','away_team','result','home_score','away_score']].copy()
backtest_2022['lambda_home'] = lambda_h_2022
backtest_2022['lambda_away'] = lambda_a_2022
backtest_2022[['p_home_win','p_draw','p_away_win']] = p_blend_2022
backtest_2022['pred'] = [INV_TARGET_MAP[i] for i in p_blend_2022.argmax(axis=1)]
display(backtest_2022.head(20))
backtest_2022.to_csv(DATA_DIR / 'wc2022_backtest_predictions_poisson_lgbm.csv', index=False)


[I 2026-06-02 15:05:49,770] A new study created in memory with name: no-name-3cb09a56-cf51-4475-b9b3-fad8b49f3dbe


  0%|          | 0/80 [00:00<?, ?it/s]

In [ ]:
# Final training for WC2026.
# Uses all known historical rows before WC2026 start, including WC2026 qualifiers from Excel if present.
final_history = history_all[history_all['date'] < WC2026_START].copy()
final_features = features_all[features_all['date'] < WC2026_START].copy()
print(final_history.shape, final_features.shape)
print(final_features['has_market_odds'].mean())

# Retune on final data. You can reuse study_2022.best_params if you want faster execution.
study_final = tune_lgbm_optuna(final_features, FEATURE_COLS, n_trials=N_OPTUNA_TRIALS, valid_start='2023-01-01')
lgbm_final = fit_lgbm_final(final_features, FEATURE_COLS, study_final.best_params, valid_start='2023-01-01')
poisson_final = fit_poisson_models(final_features, FEATURE_COLS)

# Conservative blend weight from WC2022 backtest. You can also set manually, e.g. 0.70.
FINAL_BLEND_WEIGHT_LGBM = float(best_w_2022)
print('Final blend weight LGBM:', FINAL_BLEND_WEIGHT_LGBM)


In [ ]:
# Predict WC2026 group stage.
wc2026_fixtures = pd.read_csv(WC2026_FIXTURES_WITH_ODDS_PATH)
wc2026_fixtures['date'] = pd.to_datetime(wc2026_fixtures['date'])
wc2026_fixtures = canonicalize_team_cols(wc2026_fixtures, ['home_team','away_team'])
wc2026_fixtures['tournament'] = 'FIFA World Cup'
wc2026_fixtures['neutral'] = 1
wc2026_features = build_scoring_features(final_history, wc2026_fixtures, freeze_at_min_score_date=True)

p_lgbm_2026 = lgbm_final.predict(wc2026_features[FEATURE_COLS], num_iteration=lgbm_final.best_iteration)
p_poisson_2026, lambda_h_2026, lambda_a_2026 = predict_poisson(poisson_final, wc2026_features, FEATURE_COLS)
p_blend_2026 = blend_probs(p_lgbm_2026, p_poisson_2026, FINAL_BLEND_WEIGHT_LGBM)

submission_2026_group = wc2026_features[['match_id','date','group','home_team','away_team','venue']].copy()
submission_2026_group['lambda_home'] = lambda_h_2026
submission_2026_group['lambda_away'] = lambda_a_2026
submission_2026_group[['p_home_win','p_draw','p_away_win']] = p_blend_2026
submission_2026_group['pred'] = [INV_TARGET_MAP[i] for i in p_blend_2026.argmax(axis=1)]

out_path = DATA_DIR / 'submission_wc2026_group_stage_poisson_lgbm_optuna.csv'
submission_2026_group.to_csv(out_path, index=False)
print(out_path)
display(submission_2026_group.head(20))


In [ ]:
# Feature importance from final LGBM.
importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance_gain': lgbm_final.feature_importance(importance_type='gain'),
    'importance_split': lgbm_final.feature_importance(importance_type='split'),
}).sort_values('importance_gain', ascending=False)
display(importance.head(40))
importance.to_csv(DATA_DIR / 'feature_importance_lgbm_optuna.csv', index=False)


In [ ]:
# Optional Monte Carlo group simulation and simplified bracket simulation.
# This is useful if you want champion probabilities, not just match probabilities.

def sample_score_conditional(lambda_h, lambda_a, desired_outcome, max_tries=100):
    # desired_outcome: 0 H, 1 D, 2 A
    for _ in range(max_tries):
        hg = np.random.poisson(lambda_h)
        ag = np.random.poisson(lambda_a)
        out = 0 if hg > ag else 1 if hg == ag else 2
        if out == desired_outcome:
            return int(hg), int(ag)
    # fallback deterministic minimal score
    if desired_outcome == 0: return 1, 0
    if desired_outcome == 2: return 0, 1
    return 1, 1


def simulate_group_stage(pred_df, n_sims=5000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    teams = sorted(set(pred_df['home_team']).union(pred_df['away_team']))
    qualification_counts = defaultdict(int)
    group_rank_counts = defaultdict(lambda: defaultdict(int))
    all_group_tables = []
    rows = pred_df.reset_index(drop=True)
    for sim in range(n_sims):
        table = {t: {'team':t, 'group':None, 'points':0, 'gf':0, 'ga':0, 'gd':0, 'wins':0, 'draws':0, 'losses':0} for t in teams}
        for _, r in rows.iterrows():
            h, a = r['home_team'], r['away_team']
            probs = np.array([r['p_home_win'], r['p_draw'], r['p_away_win']], dtype=float)
            outcome = rng.choice([0,1,2], p=probs/probs.sum())
            hg, ag = sample_score_conditional(r['lambda_home'], r['lambda_away'], outcome)
            for t, gf, ga in [(h,hg,ag),(a,ag,hg)]:
                table[t]['group'] = r['group']
                table[t]['gf'] += gf; table[t]['ga'] += ga; table[t]['gd'] += gf-ga
            if hg > ag:
                table[h]['points'] += 3; table[h]['wins'] += 1; table[a]['losses'] += 1
            elif hg < ag:
                table[a]['points'] += 3; table[a]['wins'] += 1; table[h]['losses'] += 1
            else:
                table[h]['points'] += 1; table[a]['points'] += 1; table[h]['draws'] += 1; table[a]['draws'] += 1
        sim_table = pd.DataFrame(table.values())
        # rank within group, random tie-breaker added after standard fields
        sim_table['rand'] = rng.random(len(sim_table))
        sim_table = sim_table.sort_values(['group','points','gd','gf','wins','rand'], ascending=[True,False,False,False,False,False])
        sim_table['rank'] = sim_table.groupby('group').cumcount() + 1
        third = sim_table[sim_table['rank'] == 3].sort_values(['points','gd','gf','wins','rand'], ascending=[False,False,False,False,False]).head(8)
        qualified = pd.concat([sim_table[sim_table['rank'] <= 2], third], ignore_index=True)
        for _, rr in qualified.iterrows():
            qualification_counts[rr['team']] += 1
        for _, rr in sim_table.iterrows():
            group_rank_counts[rr['team']][int(rr['rank'])] += 1
        if sim < 20:  # keep sample tables only, not all to save memory
            all_group_tables.append(sim_table)
    qual = pd.DataFrame({'team': list(qualification_counts.keys()), 'qualification_prob': [v/n_sims for v in qualification_counts.values()]})
    ranks = []
    for team, d in group_rank_counts.items():
        row = {'team': team}
        for r in range(1,5):
            row[f'group_rank_{r}_prob'] = d[r] / n_sims
        ranks.append(row)
    ranks = pd.DataFrame(ranks)
    return qual.merge(ranks, on='team', how='outer').sort_values('qualification_prob', ascending=False)

qualification_probs = simulate_group_stage(submission_2026_group, n_sims=5000)
display(qualification_probs.head(48))
qualification_probs.to_csv(DATA_DIR / 'wc2026_group_qualification_probabilities.csv', index=False)


## Notes de performance

- Augmente `N_OPTUNA_TRIALS` à 200–500 si tu as le temps.
- Le backtest WC2022 est le vrai sanity check. Si le log loss WC2022 se dégrade en ajoutant les odds, vérifie le mapping des odds et la date de disponibilité.
- Les features de marché doivent rester pré-match. Ne jamais utiliser `HS`, `AS`, `HST`, `HxG`, corners, cartons, etc. pour prédire un match avant coup.
- Si ton environnement n'a pas assez de mémoire, baisse `LGBM_NUM_BOOST_ROUND` ou `N_OPTUNA_TRIALS`.
